In [6]:
import pandas as pd 
import os

In [2]:
df = pd.read_csv('D:\\Portfolio\\DA\\Loan\\Data_Raw\\Loan_default.csv')

In [3]:
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default,Loan Date (DD/MM/YYYY)
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0,10/15/2018
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0,3/25/2016
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1,11/11/2013
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0,6/22/2017
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0,6/9/2014


In [9]:
import pandas as pd
import numpy as np
import os

# 1. Đọc dữ liệu từ file của bạn (Hãy đổi tên file cho đúng)
# df = pd.read_csv('your_data.csv')

# --- Tạo thư mục 'data_clean' nếu chưa có ---
output_folder = 'data_clean'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# 2. Xử lý DimDate và xuất file
df['Loan Date (DD/MM/YYYY)'] = pd.to_datetime(df['Loan Date (DD/MM/YYYY)'], dayfirst=True)
dim_date = pd.DataFrame({'FullDate': sorted(df['Loan Date (DD/MM/YYYY)'].unique())})
dim_date['DateKey'] = range(1, len(dim_date) + 1)
dim_date['Day'] = dim_date['FullDate'].dt.day
dim_date['Month'] = dim_date['FullDate'].dt.month
dim_date['Quarter'] = dim_date['FullDate'].dt.quarter
dim_date['Year'] = dim_date['FullDate'].dt.year

# Xuất file DimDate.csv vào folder
dim_date.to_csv(os.path.join(output_folder, 'DimDate.csv'), index=False)

# 3. Tạo và xuất các bảng Dimension khác
# Khai báo cấu trúc các bảng Dim cần tách
dim_configs = {
    'DimEducation': ('Education', 'EducationKey'),
    'DimEmploymentType': ('EmploymentType', 'EmploymentTypeKey'),
    'DimMaritalStatus': ('MaritalStatus', 'MaritalStatusKey'),
    'DimHasMortgage': ('HasMortgage', 'HasMortgageKey'),
    'DimHasDependents': ('HasDependents', 'HasDependentsKey'),
    'DimLoanPurpose': ('LoanPurpose', 'LoanPurposeKey'),
    'DimHasCoSigner': ('HasCoSigner', 'HasCoSignerKey')
}

temp_dims = {}
for file_name, (col_name, key_name) in dim_configs.items():
    # Tạo bảng Dim
    unique_vals = sorted(df[col_name].unique())
    d_df = pd.DataFrame({col_name: unique_vals, key_name: range(1, len(unique_vals) + 1)})
    temp_dims[file_name] = d_df
    
    # XUẤT FILE: Mỗi vòng lặp sẽ xuất 1 file CSV riêng vào folder data_clean
    d_df.to_csv(os.path.join(output_folder, f'{file_name}.csv'), index=False)

# 4. Xử lý bảng Fact và xuất file cuối cùng
fact_loan = df.copy()

# Thêm các cột Bins/Groups như sơ đồ yêu cầu
fact_loan['Age Groups'] = pd.cut(fact_loan['Age'], bins=[0, 30, 45, 60, 100], labels=['18-30', '31-45', '46-60', '60+'])
fact_loan['CreditScoreBins'] = pd.cut(fact_loan['CreditScore'], bins=[300, 580, 670, 740, 800, 850], labels=['Poor', 'Fair', 'Good', 'Very Good', 'Exceptional'])

# Map các Key từ bảng Dim vào Fact
fact_loan = fact_loan.merge(dim_date[['FullDate', 'DateKey']], left_on='Loan Date (DD/MM/YYYY)', right_on='FullDate')
for file_name, d_df in temp_dims.items():
    c_name = d_df.columns[0]
    fact_loan = fact_loan.merge(d_df, on=c_name).drop(columns=[c_name])

# Lọc các cột cho bảng Fact (giữ lại ID và các cột số)
fact_cols = [
    'LoanID', 'Age', 'Age Groups', 'Income', 'LoanAmount', 'CreditScore', 'CreditScoreBins',
    'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Default',
    'DateKey', 'EducationKey', 'EmploymentTypeKey', 'MaritalStatusKey', 
    'HasMortgageKey', 'HasDependentsKey', 'LoanPurposeKey', 'HasCoSignerKey'
]
fact_loan = fact_loan[[c for c in fact_cols if c in fact_loan.columns]]

# Xuất file FactLoan.csv vào folder
fact_loan.to_csv(os.path.join(output_folder, 'FactLoan.csv'), index=False)

print(f"Hoàn thành! Hãy kiểm tra thư mục '{output_folder}' để thấy tất cả các file CSV.")

C:\Users\Dell\AppData\Local\Temp\ipykernel_23568\1756509029.py:14: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Loan Date (DD/MM/YYYY)'] = pd.to_datetime(df['Loan Date (DD/MM/YYYY)'], dayfirst=True)


Hoàn thành! Hãy kiểm tra thư mục 'data_clean' để thấy tất cả các file CSV.
